# NLP Complete Workflow
### Go-to cheatsheet voor elk NLP project

---

## Overzicht

```
STAP 1  Imports
STAP 2  Data inladen en bekijken
STAP 3  Train/Test Split  VOOR preprocessing!
STAP 4  Preprocessing
STAP 5  Tokenization (optioneel)
STAP 6  EDA - top woorden per klasse
STAP 7  Feature Extraction
        7a BOW
        7b TF-IDF
        7c N-grams
        7d Extra features
        7e Word2Vec / CBOW / Embeddings
STAP 8  Model trainen
        8a Klassiek ML
        8b Neural Network
STAP 9  Evaluatie
STAP 10 Overfit check
```

---

## Wanneer wat gebruiken?

| Situatie | Feature Extraction | Model |
|---|---|---|
| Weinig data, snel resultaat | BOW | MultinomialNB |
| Tekstclassificatie algemeen | TF-IDF | LogisticRegression |
| Context van meerdere woorden | N-grams | LogisticRegression |
| Grote dataset, betekenis belangrijk | Word2Vec of CBOW | LR of NN |
| Volgorde van woorden belangrijk | Embedding laag | LSTM of Dense NN |

---

## PAS AAN aan jouw project
Zoek naar PAD AAN en vervang TEXT_COL, LABEL_COL en het bestandspad

## STAP 1 - Imports

**Wat:** alle benodigde libraries laden

**Waarom:** alles in een keer laden zodat je later niets vergeet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, LSTM, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer as KerasTokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from gensim.models import Word2Vec

print('Setup klaar!')

## STAP 2 - Data Inladen en Bekijken

**Wat:** data inladen en eerste blik nemen

**Waarom altijd doen:** je moet begrijpen wat in de tekst zit VOOR preprocessing.
- HTML tags? HTML cleaning nodig
- Lege waarden? fillna nodig
- Ongebalanceerde klassen? class_weight overwegen

In [ ]:
# PAS AAN
TEXT_COL  = 'text'
LABEL_COL = 'label'

df = pd.read_csv('pad/naar/jouw/data.csv', encoding='latin-1')  # PAS AAN

# Optioneel: verklein dataset
# df = df.head(1000)

print('Shape:', df.shape)
print('Kolommen:', df.columns.tolist())
print('Datatypes:')
print(df.dtypes)
print('Lege waarden:')
print(df.isnull().sum())
df.head()

In [ ]:
# Bekijk ruwe tekst - ALTIJD doen!
# Dit toont wat preprocessing nodig is
print('VOORBEELD TEKSTEN')
for i in range(3):
    print(f'[{i}]', df[TEXT_COL].iloc[i][:300])
    print('---')

In [ ]:
# Klasse distributie
# Ongebalanceerd = model kan vertekend zijn
print(df[LABEL_COL].value_counts())
df[LABEL_COL].value_counts().plot(kind='bar')
plt.title('Klasse distributie')
plt.show()

df.fillna('', inplace=True)

## STAP 3 - Train/Test Split

**Wat:** data splitsen

**Waarom VOOR preprocessing:** anders data leakage - model heeft testdata al gezien

```
X_train (80%) model leert hiervan
X_test  (20%) model heeft dit NOOIT gezien
```

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL],
    df[LABEL_COL],
    test_size=0.2,
    random_state=42
)

print('Train:', X_train.shape[0])
print('Test: ', X_test.shape[0])

## STAP 4 - Preprocessing

**Wat:** tekst opkuisen

**Waarom:** ruwe tekst bevat ruis die het model verwarren

**Volgorde is BELANGRIJK:**
```
1 HTML cleaning   weglaten als: geen HTML
2 Prefixed b      weglaten als: geen bytes strings
3 Lowercase       MOET voor stopwords!   weglaten als: hoofdletters belangrijk
4 Special chars   weglaten als: symbolen belangrijk
5 Numbers         weglaten als: cijfers betekenis hebben
6 Single chars    zelden weglaten
7 Extra spaces    zelden weglaten
8 Stopwords       MOET na lowercase!     weglaten als: grammatica belangrijk
9 Lemmatization   MOET voor features!    weglaten als: snelheid kritisch
```

# mini preprocessing pipeline, alles eerst importeren zie blok hieronder

def text_preprocessing_pipeline(text):
    # Step 1: Tokenize the text
    tokens = word_tokenize(text)

    # Step 2: Remove stop words
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]

    # Step 3: Remove punctuation
    filtered_tokens = [word for word in filtered_tokens if word not in string.punctuation]

    # Step 4: Apply lemmatization
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]
    return lemmatized_tokens


In [ ]:
# PAS AAN: taal van jouw data
stop_words = set(stopwords.words('english'))  # dutch, french, german...
lemmatizer = WordNetLemmatizer()
stemmer    = PorterStemmer()

def preprocess(text):
    # 1. HTML cleaning - weglaten als: geen HTML
    text = re.sub(r'<(script|style).*?>.*?</\1>', '', text, flags=re.DOTALL)
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)
    text = re.sub(r'<.*?>', '', text)

    # 2. Prefixed b - weglaten als: geen bytes strings
    text = re.sub(r'^b\s+', '', text)

    # 3. Lowercase - MOET voor stopwords! - weglaten als: hoofdletters betekenis
    text = text.lower()

    # 4. Special characters - weglaten als: symbolen belangrijk
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # 5. Numbers - weglaten als: cijfers betekenis hebben
    text = re.sub(r'\d+', ' ', text)

    # 6. Losse letters
    text = re.sub(r'\b[a-zA-Z]\b', ' ', text)
    text = re.sub(r'^\s*[a-zA-Z]\s+', ' ', text)

    # 7. Dubbele spaties
    text = re.sub(r'\s+', ' ', text).strip()

    # 8. Tokenization - MOET voor stopwords en lemmatization!
    # Splitst tekst in lijst van individuele woorden
    tokens = word_tokenize(text)

    # 9. Punctuation removal op token niveau
    tokens = [w for w in tokens if w not in string.punctuation]

    # 10. Stopwords - MOET na lowercase EN na tokenization!
    # weglaten als: grammatica of woordvolgorde belangrijk
    tokens = [w for w in tokens if w not in stop_words]

    # 11. Lemmatization - werkt op individuele tokens!
    # MOET na tokenization, MOET voor feature extraction!
    # alternatief: stemmer.stem(w) sneller maar ruwer
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    # 12. Terug samenvoegen naar string
    # Waarom: BOW en TF-IDF verwachten strings als input
    return ' '.join(tokens)

# Toepassen - apart op train en test
X_train = X_train.apply(preprocess)
X_test  = X_test.apply(preprocess)

print('Na preprocessing:')
print(X_train.iloc[0])

## STAP 5 - POS Tagging (optioneel)

**Wat:** elk woord een grammaticale tag geven

**Wanneer nodig:** als je alleen bepaalde woordsoorten wil behouden

**Wanneer niet nodig:** bij de meeste classificatietaken

**Let op:** tokenization, stopwords en lemmatization zitten al in de preprocess functie van stap 4!

In [ ]:
# POS Tagging (optioneel)
# Waarom: grammaticale tag per woord
# Wanneer nuttig: als je alleen bepaalde woordsoorten wil behouden
#
# NN  = zelfstandig naamwoord enkelvoud
# NNS = zelfstandig naamwoord meervoud
# VB  = werkwoord
# JJ  = bijvoeglijk naamwoord

sample = word_tokenize('the quick brown fox jumps')
print('POS tags:', pos_tag(sample))

# Alleen zelfstandige naamwoorden
def keep_nouns(tokens):
    tagged = pos_tag(tokens)
    return [w for w, tag in tagged if tag.startswith('NN')]

# Uncomment om toe te passen:
# df['tokens'] = df['tokens'].apply(keep_nouns)

## STAP 6 - EDA: Top woorden per klasse

**Wat:** meest voorkomende woorden per klasse bekijken

**Waarom:** begrijpen of klassen echt verschillende woorden gebruiken.
Als top woorden van alle klassen hetzelfde zijn - preprocessing niet goed genoeg

In [ ]:
CLASSES = df[LABEL_COL].unique()
print('Klassen:', CLASSES)

cv_eda = CountVectorizer()

for klasse in CLASSES:
    teksten = df[df[LABEL_COL] == klasse][TEXT_COL].apply(preprocess)
    matrix  = cv_eda.fit_transform(teksten)
    counts  = pd.DataFrame(matrix.sum(axis=0), columns=cv_eda.get_feature_names_out()).T
    top10   = counts.nlargest(10, 0)
    print(f'\nTOP 10 woorden voor klasse [{klasse}]:')
    print(top10)

## STAP 7 - Feature Extraction

**Wat:** tekst naar cijfers

**Waarom:** ML modellen kunnen geen tekst lezen

### 7a. Bag of Words

**Wat:** telt hoe vaak elk woord voorkomt

**Wanneer gebruiken:** eenvoudig, snel, goed als woordfrequentie een indicator is

**Wanneer niet:** volgorde of betekenis van woorden is belangrijk

```
hello world hello   naar   {hello: 2, world: 1}
```

fit_transform op train, transform op test - nooit andersom!

In [ ]:
cv = CountVectorizer()
X_train_bow = cv.fit_transform(X_train)  # leert vocabulaire uit train
X_test_bow  = cv.transform(X_test)       # gebruikt zelfde vocabulaire

print('BOW shape:', X_train_bow.shape)
print('Vocabulaire grootte:', len(cv.vocabulary_))

### 7b. TF-IDF

**Wat:** zoals BOW maar woorden die overal voorkomen krijgen minder gewicht

**Wanneer gebruiken:** slimmer dan BOW - unieke woorden per klasse krijgen meer gewicht

**Wanneer niet:** woordfrequentie is op zich al een sterke indicator

```
the - overal    lage waarde
unique_word - zelden    hoge waarde
```

In [ ]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print('TF-IDF shape:', X_train_tfidf.shape)

### 7c. N-grams

**Wat:** combinaties van meerdere woorden als features

**Wanneer gebruiken:** twee woorden samen hebben andere betekenis dan apart

**Wanneer niet:** kleine dataset - te veel features

```
unigram (1): hello, world
bigram  (2): hello world
trigram (3): hello world today
```

In [ ]:
# Bigrams
cv_bigram      = CountVectorizer(ngram_range=(2, 2))
X_train_bigram = cv_bigram.fit_transform(X_train)
X_test_bigram  = cv_bigram.transform(X_test)
print('Bigram shape:', X_train_bigram.shape)

# Unigrams + Bigrams - vaak beste resultaat
cv_uni_bi      = CountVectorizer(ngram_range=(1, 2))
X_train_uni_bi = cv_uni_bi.fit_transform(X_train)
X_test_uni_bi  = cv_uni_bi.transform(X_test)
print('Unigram+Bigram shape:', X_train_uni_bi.shape)

### 7d. Extra Features (optioneel)

**Wat:** handgemaakte features naast BOW/TF-IDF

**Wanneer gebruiken:** domeinkennis die BOW niet ziet - bv tekstlengte, patronen

**Wanneer niet:** als extra features accuracy niet verbeteren

PAS AAN: definieer jouw eigen features!

In [ ]:
X_train_df = X_train.to_frame()
X_test_df  = X_test.to_frame()

for df_part in [X_train_df, X_test_df]:
    # Altijd nuttig: lengte en woordtelling
    df_part['text_len']   = df_part[TEXT_COL].apply(len)
    df_part['word_count'] = df_part[TEXT_COL].apply(lambda x: len(x.split()))

    # PAS AAN: jouw domein-specifieke features
    # Voorbeelden:
    # df_part['heeft_url']     = df_part[TEXT_COL].str.contains('http') * 1
    # df_part['mijn_patroon']  = df_part[TEXT_COL].str.contains('woord1|woord2') * 1

extra_cols  = ['text_len', 'word_count']  # PAS AAN
extra_train = X_train_df[extra_cols].values
extra_test  = X_test_df[extra_cols].values

print('Extra features shape:', extra_train.shape)
X_train_df.head()

### 7e. Word2Vec / CBOW / Embeddings

**Wat:** woorden naar vectoren met betekenis

**Waarom gebruiken:** BOW telt, embeddings begrijpen

```
BOW:       king naar [0, 0, 1, 0]    gewoon tellen
Embedding: king naar [0.2, 0.8, 0.1] bevat betekenis

king - man + woman = queen   embeddings snappen dit!
```

**CBOW sg=0:** context naar middelste woord voorspellen - snel

**Skip-gram sg=1:** woord naar context voorspellen - beter voor zeldzame woorden

**Wanneer gebruiken:** grote datasets, betekenis en context zijn belangrijk

In [ ]:
# Word2Vec trainen - alleen op traindata!
sentences = [text.split() for text in X_train]

w2v_model = Word2Vec(
    sentences,
    vector_size=100,  # grootte embedding vector
    window=5,         # context venster links en rechts
    min_count=2,      # woorden minder dan X keer worden genegeerd
    sg=0,             # 0=CBOW, 1=Skip-gram
    epochs=10
)

# Test
try:
    test_word = X_train.iloc[0].split()[0]
    print(f'Gelijkaardig aan {test_word}:')
    print(w2v_model.wv.most_similar(test_word, topn=5))
except:
    print('Woord niet gevonden')

In [ ]:
# Gemiddelde embedding per tekst
# Waarom: ML model verwacht één vector per tekst, niet per woord

VECTOR_SIZE = 100

def get_avg_embedding(text, model, vector_size=VECTOR_SIZE):
    words = text.split()
    vecs  = [model.wv[w] for w in words if w in model.wv]
    if len(vecs) == 0:
        return np.zeros(vector_size)
    return np.mean(vecs, axis=0)

X_train_w2v = np.array([get_avg_embedding(t, w2v_model) for t in X_train])
X_test_w2v  = np.array([get_avg_embedding(t, w2v_model) for t in X_test])

print('Word2Vec shape:', X_train_w2v.shape)

## STAP 8 - Model Trainen

### 8a. Klassiek ML

**Wanneer:** weinig data, snel resultaat, eenvoudige classificatie

```
MultinomialNB      snel, goed voor BOW/TF-IDF, weinig data
LogisticRegression iets beter dan NB, goed startpunt
RandomForest       complexer, trager, groot dataset
```

In [ ]:
results = {}

# BOW + MultinomialNB
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
results['BOW + NB'] = accuracy_score(y_test, nb.predict(X_test_bow))

# TF-IDF + MultinomialNB
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
results['TF-IDF + NB'] = accuracy_score(y_test, nb.predict(X_test_tfidf))

# BOW + extra features
# hstack plakt twee matrices naast elkaar
nb = MultinomialNB()
nb.fit(hstack([X_train_bow, extra_train]), y_train)
results['BOW + extra + NB'] = accuracy_score(y_test, nb.predict(hstack([X_test_bow, extra_test])))

# TF-IDF + extra features
nb = MultinomialNB()
nb.fit(hstack([X_train_tfidf, extra_train]), y_train)
results['TF-IDF + extra + NB'] = accuracy_score(y_test, nb.predict(hstack([X_test_tfidf, extra_test])))

# BOW + LogisticRegression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_bow, y_train)
results['BOW + LR'] = accuracy_score(y_test, lr.predict(X_test_bow))

# TF-IDF + LogisticRegression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
results['TF-IDF + LR'] = accuracy_score(y_test, lr.predict(X_test_tfidf))

# Word2Vec + LogisticRegression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_w2v, y_train)
results['Word2Vec + LR'] = accuracy_score(y_test, lr.predict(X_test_w2v))

print('MODEL VERGELIJKING')
for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f'{name:<30} {score:.4f}')

### 8b. Neural Network

**Wanneer:** veel data, context belangrijk, hogere accuracy nodig

```
Tekst naar integers naar Embedding laag naar Dense lagen naar klasse
```

**Embedding laag vs Word2Vec:**
- Word2Vec: apart getraind, dan gemiddeld
- Embedding laag: leert TIJDENS training - beter!

In [ ]:
# PAS AAN
VOCAB_SIZE = 10000  # max woorden in vocabulaire
MAX_LEN    = 200    # max tekstlengte
EMBED_DIM  = 64     # grootte embedding vector

# Tekst naar integers
keras_tokenizer = KerasTokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
keras_tokenizer.fit_on_texts(X_train)  # alleen op traindata!

X_train_seq = keras_tokenizer.texts_to_sequences(X_train)
X_test_seq  = keras_tokenizer.texts_to_sequences(X_test)

# Padding: alle teksten even lang maken
# Waarom: Neural Networks verwachten gelijke input lengte
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print('Shape na padding:', X_train_pad.shape)

In [ ]:
NUM_CLASSES = len(df[LABEL_COL].unique())
print('Klassen:', NUM_CLASSES)

model_nn = Sequential([
    # Embedding: leert betekenis van woorden tijdens training
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),

    # GlobalAveragePooling1D: middelt embeddings over tekst
    # Alternatief: LSTM() als woordvolgorde belangrijk
    GlobalAveragePooling1D(),

    Dense(64, activation='relu'),
    Dropout(0.3),  # 30% neuronen uitschakelen: tegen overfit

    # PAS AAN: output laag
    # 2 klassen:  Dense(1, activation='sigmoid')
    # 3+ klassen: Dense(NUM_CLASSES, activation='softmax')
    Dense(1, activation='sigmoid')
])

model_nn.compile(
    optimizer='adam',
    # PAS AAN: loss
    # 2 klassen:  binary_crossentropy
    # 3+ integer: sparse_categorical_crossentropy
    # 3+ one-hot: categorical_crossentropy
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_nn.summary()

In [ ]:
# EarlyStopping: stop als val_loss niet meer daalt - voorkomt overfit
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model_nn.fit(
    X_train_pad, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

## STAP 9 - Evaluatie

**Wat:** hoe goed presteert het model?

**Waarom niet alleen accuracy:** model dat altijd meest voorkomende klasse voorspelt heeft hoge accuracy maar is nutteloos

```
Precision  van alle voorspeld als X, hoeveel waren echt X?
Recall     van alle echte X, hoeveel correct gevonden?
F1         gemiddelde van precision en recall
```

In [ ]:
# PAS AAN: beste model en feature set
best_model   = nb
X_train_best = X_train_bow
X_test_best  = X_test_bow

y_pred = best_model.predict(X_test_best)

print('Accuracy:', accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

# Confusion matrix: diagonaal = correct, buiten = fouten
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Neural Network training curves
# Overfit: train veel hoger dan val
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validatie')
axes[0].set_title('Accuracy per epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validatie')
axes[1].set_title('Loss per epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## STAP 10 - Overfit Check

**Wat:** train vs test score vergelijken

**Waarom:** model heeft traindata onthouden maar leert niet echt

```
train=99% test=70%  OVERFIT    dropout toevoegen, minder features
train=92% test=90%  GOED
train=60% test=58%  UNDERFIT   complexer model, meer features
```

In [ ]:
y_train_pred = best_model.predict(X_train_best)
y_test_pred  = best_model.predict(X_test_best)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc  = accuracy_score(y_test,  y_test_pred)

print(f'Train accuracy: {train_acc:.4f}')
print(f'Test accuracy:  {test_acc:.4f}')
print(f'Verschil:       {train_acc - test_acc:.4f}')

if train_acc - test_acc > 0.1:
    print('Mogelijk overfit! Verschil > 10%')
else:
    print('Geen duidelijk overfit')

In [ ]:
# Cross validatie: betrouwbaardere schatting dan 1 split
scores = cross_val_score(best_model, X_train_best, y_train, cv=5)
print('CV scores:', scores.round(4))
print('Gemiddelde:', scores.mean().round(4))
print('Als gemiddelde = test accuracy: model generaliseert goed!')

## Referentie - Preprocessing checklist

| Stap | Bijna altijd? | Wanneer weglaten |
|---|---|---|
| HTML cleaning | Nee | geen HTML in data |
| Lowercase | Ja | hoofdletters hebben betekenis |
| Special chars | Ja | symbolen zijn belangrijk |
| Numbers | Nee | cijfers hebben betekenis |
| Single chars | Ja | zelden nuttig |
| Stopwords | Ja | grammatica is belangrijk |
| Lemmatization | Ja | snelheid kritisch - gebruik stemmer |

## Referentie - Feature Extraction

| Methode | Volgorde? | Betekenis? | Data nodig |
|---|---|---|---|
| BOW | Nee | Nee | Weinig |
| TF-IDF | Nee | Nee | Weinig |
| N-grams | Deels | Nee | Gemiddeld |
| Word2Vec/CBOW | Nee | Ja | Veel |
| Embedding laag NN | Ja | Ja | Veel |

## Referentie - Output laag

| Situatie | Activatie | Loss |
|---|---|---|
| 2 klassen | sigmoid | binary_crossentropy |
| 3+ klassen integers | softmax | sparse_categorical_crossentropy |
| 3+ klassen one-hot | softmax | categorical_crossentropy |